# 05 — Covenant Analysis

Проверка ковенантов по прогнозным годам.


In [ ]:
# ── Setup ──────────────────────────────────────────────
import sys, pathlib
ROOT = pathlib.Path('.').resolve()
for _p in [ROOT] + list(ROOT.parents):
    if (_p / 'engine').is_dir():
        ROOT = _p; break
sys.path.insert(0, str(ROOT))

COMPANY = None
for _p in [pathlib.Path('.').resolve()] + list(pathlib.Path('.').resolve().parents):
    if _p.parent.name == 'companies':
        COMPANY = _p.name; break
if not COMPANY: COMPANY = 'test_metals'  # fallback
print(f'Company: {COMPANY}, ROOT: {ROOT}')


In [ ]:
from engine.orchestrator import build_model
import yaml

# Load covenant thresholds from YAML
cfg_path = ROOT / 'companies' / COMPANY / 'configs' / 'project.yaml'
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

cov = cfg.get('covenants', {})
print(f'Covenants enabled: {cov.get("enabled", False)}')
print(f'Thresholds: {cov.get("thresholds", {})}')

# Run model
r = build_model(COMPANY, run_preprocessor=True, run_model=True)
mr = r.model_result
print(f'\nBS max: {max(mr.bs_diffs.values()):.2f}')

# Check covenants
thresholds = cov.get('thresholds', {})
nd_max = thresholds.get('net_debt_ebitda_max', 99)
icr_min = thresholds.get('interest_coverage_min', 0)

print(f'\n{"Year":<6}{"ND/EBITDA":>10}{"Max":>6}{"ICR":>8}{"Min":>6}{"Status":>8}')
for yr in sorted(mr.years):
    s = mr.years[yr]
    debt = abs(s.long_term_debt or 0) + abs(s.short_term_debt or 0)
    nd = debt - (s.cash or 0)
    eb = s.ebitda or 1
    ie = abs(s.interest_expense or 0) or 1
    nd_eb = nd / eb if eb > 0 else 99
    icr = eb / ie
    ok = nd_eb <= nd_max and icr >= icr_min
    print(f'{yr:<6}{nd_eb:>10.2f}{nd_max:>6.1f}{icr:>8.2f}{icr_min:>6.1f}{"  OK" if ok else "  BREACH":>8}')
